# Week 22, Day 3: VS Code and the debugger

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w022d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def average(marks):
    total = 0
    for m in marks:
        total += m
    return total / len(marks) + 1

print(average([60, 70, 80]))

In [ ]:
import sys

def average(marks):
    total = 0
    for m in marks:
        total += m
    return total / len(marks) + 1

events = []

def tracer(frame, event, arg):
    if frame.f_code.co_name == "average":
        if event == "line":
            events.append((frame.f_lineno, dict(frame.f_locals)))
        elif event == "return":
            events.append(("returned", arg))
    return tracer

sys.settrace(tracer)
average([60, 70, 80])
sys.settrace(None)

for line, info in events[-3:]:
    print(line, info)

> **Key idea.** Open the project **folder** in VS Code with the Python extension and choose the venv's interpreter. Debug with breakpoints (click the margin), F5 to start, F10 to step over, F11 to step into, and watch the Variables and Call stack panes. In a terminal, `breakpoint()` gives you `pdb`. Debuggers work by asking Python to call a trace function on every line.

### Exercise: Which lines ran?

Write `lines_run(func, *args)` that runs `func(*args)` with a trace function and returns the list of line numbers (relative to the function's first line, `def` being 0) of every `"line"` event **in that function**, in order. Use `frame.f_lineno - func.__code__.co_firstlineno`.

In [ ]:
import sys

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    def sign(x):\n        if x > 0:\n            return \"positive\"\n        if x < 0:\n            return \"negative\"\n        return \"zero\"\n    assert lines_run(sign, 5) == [1, 2]\n    assert lines_run(sign, -5) == [1, 3, 4]\n    assert lines_run(sign, 0) == [1, 3, 5]\n    import sys\n    assert sys.gettrace() is None or sys.gettrace() is not lines_run, \"turn tracing off afterwards\"")

### Exercise: Watch a variable

Write `watch(func, name, *args)` returning the list of **different** values the local variable `name` takes inside `func` while it runs, in order (record a value each time a `"line"` or `"return"` event finds it changed from the last value recorded). Copy values with `copy.copy` so a list that changes later isn't recorded as its final state.

In [ ]:
import sys
import copy

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    def average(marks):\n        total = 0\n        for m in marks:\n            total += m\n        return total / len(marks) + 1\n    assert watch(average, \"total\", [60, 70, 80]) == [0, 60, 130, 210]\n    assert watch(average, \"m\", [5, 5, 6]) == [5, 6]\n    def build(n):\n        out = []\n        for i in range(n):\n            out.append(i)\n        return out\n    assert watch(build, \"out\", 3) == [[], [0], [0, 1], [0, 1, 2]]")

### Exercise: The call stack

Write `call_log(func, *args)` returning a list of strings showing every call and return of functions defined in the **same module as `func`**, indented two spaces per level of depth: `"fact(3)"` style isn't available from frames, so use `"call NAME"` and `"return NAME -> VALUE"` (with `repr` of the value). Start at depth 0 for the first call.

In [ ]:
import sys

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    def fact(n):\n        return 1 if n <= 1 else n * fact(n - 1)\n    assert call_log(fact, 3) == [\"call fact\", \"  call fact\", \"    call fact\", \"    return fact -> 1\", \"  return fact -> 2\", \"return fact -> 6\"]")

### Exercise: A conditional breakpoint

Write `stop_when(func, condition, *args)` that runs `func(*args)` and, the **first** time a `"line"` event in `func` finds `condition(locals)` true (where `locals` is the frame's local variables), records the relative line number and a copy of the locals, then stops tracing. Return `(line, locals)` or `None` if the condition never held. A `condition` that raises an error (because a variable doesn't exist yet) counts as false.

In [ ]:
import sys

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    def process(items):\n        total = 0\n        for i, x in enumerate(items):\n            total += 100 // x\n        return total\n    hit = stop_when(process, lambda v: v[\"x\"] == 0, [5, 4, 0, 2])\n    assert hit == (3, {\"items\": [5, 4, 0, 2], \"total\": 45, \"i\": 2, \"x\": 0})\n    assert stop_when(process, lambda v: v[\"total\"] > 10 ** 6, [5]) is None")

### Exercise: Know your keys

Fill in the VS Code action for each key (on Windows and Linux), choosing from: `"command palette"`, `"step over"`, `"step into"`, `"go to definition"`, `"rename symbol"`, `"start debugging"`.

In [ ]:
keys = {
    "Ctrl+Shift+P": ___,
    "F5": ___,
    "F10": ___,
    "F11": ___,
    "F12": ___,
    "F2": ___,
}

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert keys == {\"Ctrl+Shift+P\": \"command palette\", \"F5\": \"start debugging\", \"F10\": \"step over\", \"F11\": \"step into\", \"F12\": \"go to definition\", \"F2\": \"rename symbol\"}")

### Bug hunt: Use the evidence

`average` is wrong. Run the tracing code below it: the watched values of `total` are right, so the loop is fine. Fix only the line the evidence points to, so it prints `[0, 60, 130, 210]` and then `70.0`.

In [ ]:
import sys

def average(marks):
    total = 0
    for m in marks:
        total += m
    return total / len(marks) + 1

seen = []
def tracer(frame, event, arg):
    if frame.f_code.co_name == "average" and event in ("line", "return"):
        v = frame.f_locals.get("total")
        if v is not None and (not seen or seen[-1] != v):
            seen.append(v)
    return tracer

sys.settrace(tracer)
result = average([60, 70, 80])
sys.settrace(None)
print(seen)
print(result)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"[0, 60, 130, 210]\\n70.0\")")

### Bug hunt: A forgotten breakpoint

This script was committed with a `breakpoint()` left in, so on a real computer it stops and waits at a `(Pdb)` prompt every time it runs. (On this site, debugging prompts are turned off, so it just runs.) Remove the debugging leftovers so the program is clean: no `breakpoint()` and no `import pdb`, and it still prints `15`.

In [ ]:
import pdb

def total(xs):
    result = 0
    for x in xs:
        breakpoint()
        result += x
    return result

print(total([1, 2, 3, 4, 5]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"clean\"\n    import os\n    os.environ[\"PYTHONBREAKPOINT\"] = \"0\"\n    expect_output(\"15\")\n    assert \"breakpoint()\" not in source and \"import pdb\" not in source")

### Puzzle: A tiny tracer

Arrange the lines so the program prints each line number `square` runs (relative to its `def`). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # the built-in function that pauses a script in the pdb debugger
b = ___   # the pdb command that prints a value (one letter)
c = ___   # True or False: a breakpoint pauses AFTER its line has run
d = ___   # the frame attribute holding the current line number

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a in (\"breakpoint\", \"breakpoint()\") and b == \"p\" and c is False and d == \"f_lineno\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a line-by-line profiler

Debuggers' cousins are **profilers**, which measure where a program spends its time. Write `line_counts(func, *args)` that runs the function with a trace function and returns a dictionary from each relative line number in `func` (as before, `def` is 0) to how many times it ran. Then `hottest(func, *args)` returning the relative line number that ran most often (the smallest number if tied), and `report(func, source, *args)` returning one string per line of the function's source text `source` (on your own computer you'd get it with `inspect.getsource(func)`; here it's passed in), formatted as the count right-aligned in 6 characters (blank for lines that never ran), two spaces, then the source line.

In [ ]:
import sys
import inspect

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_counts():\n    \"counts\"\n    def pairs(n):\n        count = 0\n        for i in range(n):\n            for j in range(i):\n                count += 1\n        return count\n    c = line_counts(pairs, 5)\n    assert c[1] == 1 and c[5] == 1 and c[4] == 10\n    assert hottest(pairs, 5) == 3\n    text = \"def pairs(n):\\n    count = 0\\n    for i in range(n):\\n        for j in range(i):\\n            count += 1\\n    return count\\n\"\n    lines = report(pairs, text, 3)\n    assert lines[0] == \"        def pairs(n):\"\n    assert lines[4] == \"     3              count += 1\"\n    assert lines[5] == \"     1      return count\"")

## Warm-up

The answer should be 70, and it's 71. In a three-line function you can spot why. In a three-thousand-line program you can't, and that's where a debugger earns its keep: it lets you stop the program on any line and look at every variable, instead of guessing and adding `print` calls. This site's X-ray does the same thing for the code on a page; today you'll get the real thing for your own projects.

## Setting up VS Code

1. Download VS Code from code.visualstudio.com and install it.
2. Open it, click the **Extensions** icon on the left (four squares), search for **Python** and install the one published by Microsoft. It brings in **Pylance** (code intelligence) and the **debugger**.
3. **File, Open Folder**, and choose your project folder (say `atlas/week22`). Always open the **folder**, not a single file: settings, the terminal and the debugger all work from the folder.
4. **Choose the interpreter:** open the command palette (**Ctrl+Shift+P**, or **Cmd+Shift+P** on a Mac), type "Python: Select Interpreter", and pick the one inside your project's `.venv`. Now running, debugging and the terminal all use that environment.
5. Open the built-in terminal with **Ctrl+`** (the key above Tab). It starts in your project folder with the venv already active.
6. Make `hello.py`, write a line, and click the **Run** triangle at the top right, or press **F5** to run with the debugger.

## Editor time-savers

| Feature | How | Why |
|---|---|---|
| Command palette | Ctrl+Shift+P | every command, searchable: you never need to remember menus |
| Completion | just type; Tab or Enter to accept | names, methods and arguments, with documentation |
| Hover | point at a name | its type, signature and docstring |
| Go to definition | F12, or Ctrl+click | jump to where a function or class is defined |
| Find all references | Shift+F12 | everywhere a name is used |
| Rename symbol | F2 | renames a variable or function everywhere, correctly (unlike find-and-replace) |
| Problems | squiggly underlines; Ctrl+Shift+M lists them | mistakes found as you type |
| Quick open | Ctrl+P, then a file name | open any file in the project |
| Multi-cursor | Alt+click, or Ctrl+D to add the next match | edit several places at once |
| Format document | Shift+Alt+F | tidy the layout (tomorrow) |
| Toggle comment | Ctrl+/ | comment or uncomment the selected lines |

On a Mac, use Cmd where the table says Ctrl. Learn two or three at a time, until you use them without thinking.

## The debugger

To debug a program:

1. **Set a breakpoint**: click in the margin to the left of a line number. A red dot appears. The program will pause **before** running that line.
2. Press **F5** (or Run, Start Debugging), and choose "Python File" if asked.
3. When it pauses, the line is highlighted, and the **Run and Debug** panel shows:
   - **Variables**: every local and global name and its value; expand lists and dictionaries to look inside.
   - **Watch**: expressions you want to keep an eye on, such as `total / len(marks)`.
   - **Call stack**: which function called which to get here (Week 15's frames).
4. Move on with the toolbar:

| Button | Key | Does |
|---|---|---|
| Continue | F5 | run until the next breakpoint |
| Step over | F10 | run this line, then pause on the next |
| Step into | F11 | if this line calls your function, go inside it |
| Step out | Shift+F11 | finish this function, pause where it was called |
| Restart | Ctrl+Shift+F5 | start again |
| Stop | Shift+F5 | end the session |

**Conditional breakpoints**: right-click a breakpoint, choose "Edit Breakpoint", and give a condition such as `m > 100`: it only pauses when that's true. Invaluable in a loop that goes wrong on its 5,000th time round.

**The debug console** lets you type any expression while paused, and see its value, using the program's variables.

The way to use a debugger on a bug: form a guess ("I think `total` is wrong after the loop"), put a breakpoint where you can test it, and look. Each look either confirms the guess or tells you where to look next. It's far faster than reading the code again and again.

## pdb in the terminal

Python has a debugger built in, `pdb`, which works anywhere, even on a server with no editor. Put this line where you want to pause:

```
breakpoint()
```

Run the script from the terminal, and when it reaches that line you get a `(Pdb)` prompt:

| Command | Does |
|---|---|
| `p expression` | print a value: `p total` |
| `n` | next line (step over) |
| `s` | step into a call |
| `c` | continue to the next breakpoint |
| `l` | list the code around the current line |
| `w` | where: show the call stack |
| `q` | quit |

Remove `breakpoint()` lines before you commit. (VS Code also stops at `breakpoint()` when debugging.)

## How a debugger works

How can one program pause another, line by line? Python lets you register a **trace function** with `sys.settrace`. Python then calls it as the program runs: when a function is called (`"call"`), before each line (`"line"`), and when a function returns (`"return"`). The trace function gets the current **frame**, which holds the line number (`frame.f_lineno`), the function's name (`frame.f_code.co_name`) and its local variables (`frame.f_locals`). A debugger is a trace function that, at a breakpoint, stops and waits for you.

The last line run before returning had `total` equal to 210, so the loop is right and the bug is in the `return`. That's debugging: look at the state, and the cause becomes obvious. This site's X-ray works this way too.

## Practice

## Quiz

## Daily challenge

## Before you go

Install VS Code and the Python extension, open your `atlas` folder, select its venv, and debug yesterday's `rich` example: put a breakpoint on the `print` line, step through, and look at the Variables pane. Tomorrow: linters and formatters, which find problems and tidy layout automatically.